# 04 · Supervised FT-Transformer
Sections 1–3 reproduce all five folds from the final checkpoints. Section 4 rebuilds the raw features and trains for 24 epochs per fold.

## 1. Data and model settings

In [ ]:
from pathlib import Path
import json
import math

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'environment/shared.ipynb').exists())
def load_cells(path):
    for cell in json.loads(path.read_text())['cells']:
        if cell['cell_type'] == 'code':
            exec(''.join(cell['source']), globals())

load_cells(ROOT / 'environment/shared.ipynb')
import torch
train = pd.read_csv(ROOT / 'data/raw/train.csv', dtype={'event': str, 'user': str})
test = pd.read_csv(ROOT / 'data/raw/test.csv', dtype={'event': str, 'user': str})
samples = pd.concat([train.drop(columns=['interested', 'not_interested']), test], ignore_index=True)
folds = pd.read_csv(ROOT / 'data/folds.csv').fold.to_numpy()
base = pd.read_pickle(ROOT / '02_xgboost_baseline/reference/features.pkl')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(4)
HERE = ROOT / '04_ft_transformer'
reference = HERE / 'reference'
output = HERE / 'results'
output.mkdir(exist_ok=True)
from sklearn.preprocessing import QuantileTransformer
from rtdl_revisiting_models import FTTransformer
CONFIG = dict(model=FTTransformer.get_default_kwargs(), seed=42, batch_size=256, fixed_epochs=24, lr=0.0001, weight_decay=0.00001, clip_grad_norm=1.0, dtype='float32', package='rtdl_revisiting_models==0.0.2')
y = np.r_[train.interested.to_numpy(dtype='float32'), np.zeros(len(test), dtype='float32')]

## 2. Fold preprocessing, model and optimization

In [ ]:
def preprocess(frame, fit, destination):
    cats = frame.select_dtypes(include=['object', 'string']).columns.tolist()
    nums = [column for column in frame if column not in cats]
    raw = frame[nums].to_numpy(dtype=np.float32)
    raw[~np.isfinite(raw)] = np.nan
    missing = np.isnan(raw)
    medians = np.nanmedian(raw[fit], axis=0)
    medians = np.nan_to_num(medians, nan=0.0)
    filled = np.where(missing, medians, raw)
    normalizer = QuantileTransformer(n_quantiles=min(1000, len(fit)), output_distribution='normal', subsample=None, random_state=42)
    normalizer.fit(filled[fit])
    extra = np.flatnonzero(missing[fit].any(axis=0))
    cont = np.column_stack([normalizer.transform(filled), missing[:, extra]]).astype(np.float32)
    encoded = []
    mappings = {}
    cardinalities = []
    for column in cats:
        values = frame[column].astype('string')
        counts = values.iloc[fit].value_counts()
        mapping = {value: index + 2 for index, value in enumerate(sorted(counts[counts >= 5].index.tolist()))}
        codes = values.map(mapping).fillna(1).to_numpy(dtype=np.int64)
        codes[values.isna().to_numpy()] = 0
        encoded.append(codes)
        mappings[column] = mapping
        cardinalities.append(len(mapping) + 2)
    cat = np.column_stack(encoded)
    pd.to_pickle(dict(numeric_columns=nums, categorical_columns=cats, medians=medians, missing_indicator_indices=extra, normalizer=normalizer, mappings=mappings, cardinalities=cardinalities, fit_rows=fit), destination)
    return cont, cat, cardinalities
def tensors(cont, cat, y, device):
    return torch.from_numpy(cont).to(device), torch.from_numpy(cat).to(device), torch.from_numpy(y).to(device)
def make_model(numeric, cardinalities, device):
    torch.manual_seed(CONFIG['seed'])
    model = FTTransformer(n_cont_features=numeric, cat_cardinalities=cardinalities, d_out=1, **CONFIG['model']).to(device)
    return model, model.make_default_optimizer()
def epoch_train(model, optimizer, data, fit, rng):
    model.train()
    order = rng.permutation(fit)
    loss_sum = 0.0
    for start in range(0, len(order), CONFIG['batch_size']):
        index = torch.as_tensor(order[start:start + CONFIG['batch_size']], dtype=torch.long, device=data[0].device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(data[0][index], data[1][index]).squeeze(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, data[2][index])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip_grad_norm'])
        optimizer.step()
        loss_sum += float(loss.detach().item()) * len(index)
    return loss_sum / len(fit)
def predict(model, data, indices):
    model.eval()
    scores = []
    with torch.inference_mode():
        for start in range(0, len(indices), CONFIG['batch_size']):
            index = torch.as_tensor(indices[start:start + CONFIG['batch_size']], dtype=torch.long, device=data[0].device)
            scores.append(model(data[0][index], data[1][index]).squeeze(-1).sigmoid().cpu().numpy())
    return np.concatenate(scores)

## 3. Run inference on every held-out user

In [ ]:
def apply_preprocessor(frame, state):
    raw = frame[state['numeric_columns']].to_numpy(dtype=np.float32)
    missing = ~np.isfinite(raw)
    filled = np.where(missing, state['medians'], raw)
    cont = np.column_stack([state['normalizer'].transform(filled), missing[:, state['missing_indicator_indices']]]).astype(np.float32)
    categories = []
    for name in state['categorical_columns']:
        values = frame[name].astype('string')
        codes = values.map(state['mappings'][name]).fillna(1).to_numpy(dtype=np.int64)
        codes[values.isna().to_numpy()] = 0
        categories.append(codes)
    return cont, np.column_stack(categories)

scores = np.zeros(len(train))
for fold in range(1, 6):
    valid = np.flatnonzero(folds == fold)
    state = pd.read_pickle(reference / f'fold_{fold}_preprocessor.pkl')
    cont, cat = apply_preprocessor(base, state)
    model, _ = make_model(cont.shape[1], state['cardinalities'], device)
    model.load_state_dict(torch.load(reference / f'fold_{fold}.pt', map_location=device))
    scores[valid] = predict(model, tensors(cont, cat, y, device), valid)
expected = pd.read_csv(reference / 'oof.csv').probability
print('OOF MAP@200', map_users(train, scores).ap200.mean())
print('Maximum probability difference', float(np.abs(scores - expected).max()))

## 4. Rebuild features and train each fold for 24 epochs

In [ ]:
x, _ = features(ROOT / 'data/raw', samples)
oof = np.zeros(len(train))
test_scores = np.zeros(len(test))
for fold in range(1, 6):
    fit, valid = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
    cont, cat, cards = preprocess(x, fit, output / f'fold_{fold}_preprocessor.pkl')
    model, optimizer = make_model(cont.shape[1], cards, device)
    data = tensors(cont, cat, y, device)
    rng = np.random.default_rng(CONFIG['seed'])
    for epoch in range(CONFIG['fixed_epochs']):
        loss = epoch_train(model, optimizer, data, fit, rng)
    torch.save(model.state_dict(), output / f'fold_{fold}.pt')
    oof[valid] = predict(model, data, valid)
    test_scores += predict(model, data, np.arange(len(train), len(x))) / 5
    print(fold, map_users(train.iloc[valid], oof[valid]).ap200.mean())
train[['user', 'event', 'interested']].assign(fold=folds, probability=oof).to_csv(output / 'oof.csv', index=False)
test[['user', 'event']].assign(probability=test_scores).to_csv(output / 'test_predictions.csv', index=False)
print('OOF MAP@200', map_users(train, oof).ap200.mean())